In [2]:
import pandas as pd
import matplotlib as plt


In [5]:
df_2018 = pd.read_csv("data/df_2018.csv") 
df_2019 = pd.read_csv("data/df_2019.csv")
df_2020 = pd.read_csv("data/df_2020.csv")
df_2021 = pd.read_csv("data/df_2021.csv")
df_2022 = pd.read_csv("data/df_2022.csv")
df_2022.head()

,CRASH_DATE,POSTED_SPEED_LIMIT,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,ROADWAY_SURFACE_COND,ROAD_DEFECT,CRASH_TYPE,DAMAGE,PRIM_CONTRIBUTORY_CAUSE,SEC_CONTRIBUTORY_CAUSE,STREET_NAME,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE
0,2022-01-01 00:00:00,30,SNOW,UNKNOWN,PARKED MOTOR VEHICLE,SNOW OR SLUSH,UNKNOWN,NO INJURY / DRIVE AWAY,"OVER $1,500",UNABLE TO DETERMINE,UNABLE TO DETERMINE,COTTAGE GROVE AVE,0,7,1,41.783176,-87.605844
1,2022-01-01 00:00:00,30,UNKNOWN,DARKNESS,PARKED MOTOR VEHICLE,UNKNOWN,UNKNOWN,INJURY AND / OR TOW DUE TO CRASH,"OVER $1,500",UNABLE TO DETERMINE,UNABLE TO DETERMINE,LAVERGNE AVE,0,7,1,41.878435,-87.749197
2,2022-01-01 00:01:00,30,RAIN,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,WET,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,"OVER $1,500",UNABLE TO DETERMINE,NOT APPLICABLE,ASHLAND AVE,0,7,1,41.959862,-87.669060
3,2022-01-01 00:13:00,25,CLEAR,DARKNESS,PEDESTRIAN,UNKNOWN,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,$500 OR LESS,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNABLE TO DETERMINE,DEARBORN ST,0,7,1,41.892485,-87.629824
4,2022-01-01 00:16:00,25,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,WET,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,"$501 - $1,500",FOLLOWING TOO CLOSELY,FAILING TO REDUCE SPEED TO AVOID CRASH,ASHLAND AVE,0,7,1,41.939607,-87.668512


In [7]:
dfs = [df_2018, df_2019, df_2020, df_2021, df_2022]
years = [2018, 2019, 2020, 2021, 2022]
for df, year in zip(dfs, years):
    df['Year'] = year
    df_all = pd.concat(dfs, ignore_index=True)

In [10]:
import plotly.graph_objects as go
import pandas as pd

# Causes to exclude
excluded_causes = ["UNABLE TO DETERMINE", "NOT APPLICABLE"]

# Get unique years from dataset
years = sorted(df_all["Year"].unique())

# Filter out unwanted causes
df_filtered = df_all[~df_all["PRIM_CONTRIBUTORY_CAUSE"].isin(excluded_causes)]

# Aggregate data: Count occurrences of each cause per year
df_grouped = df_filtered.groupby(["Year", "PRIM_CONTRIBUTORY_CAUSE"]).size().reset_index(name="Count")

# Function to get the top N causes per year
top_n = 8  
df_top = df_grouped.groupby("Year").apply(lambda x: x.nlargest(top_n, "Count")).reset_index(drop=True)

# Create an empty figure
fig = go.Figure()

# Store visibility settings
buttons = []

# Add a trace for each year
for year in years:
    df_year = df_top[df_top["Year"] == year]  # Get only top causes per year

    fig.add_trace(go.Bar(
        x=df_year["PRIM_CONTRIBUTORY_CAUSE"],
        y=df_year["Count"],
        name=str(year),
        visible=True  # Default: Show all traces
    ))

# Create dropdown buttons for year selection
buttons.append({
    "label": "All Years",
    "method": "update",
    "args": [{"visible": [True] * len(years)}, {"title": "Top Crash Causes for All Years"}]
})

for i, year in enumerate(years):
    visibility = [False] * len(years)  
    visibility[i] = True 

    buttons.append({
        "label": str(year),
        "method": "update",
        "args": [{"visible": visibility}, {"title": f"Top {top_n} Crash Causes in {year}"}]
    })

# Update layout with dropdown menu
fig.update_layout(
    xaxis_tickangle=-45,
    updatemenus=[{
        "buttons": buttons,
        "direction": "down",
        "showactive": True,
    }],
    barmode="group"
)
fig.show()

C:\Users\quaid\AppData\Local\Temp\ipykernel_16936\6378124.py:18: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [11]:
import plotly.graph_objects as go
import pandas as pd

df_all["CRASH_HOUR"] = df_all["CRASH_HOUR"].astype(int)
years = sorted(df_all["Year"].unique())
all_hours = pd.DataFrame({"CRASH_HOUR": range(24)})

# Mapping hour numbers (0-23) to time labels
hour_to_time = {
    0: "12:00 AM", 1: "1:00 AM", 2: "2:00 AM", 3: "3:00 AM", 4: "4:00 AM", 5: "5:00 AM",
    6: "6:00 AM", 7: "7:00 AM", 8: "8:00 AM", 9: "9:00 AM", 10: "10:00 AM", 11: "11:00 AM",
    12: "12:00 PM", 13: "1:00 PM", 14: "2:00 PM", 15: "3:00 PM", 16: "4:00 PM", 17: "5:00 PM",
    18: "6:00 PM", 19: "7:00 PM", 20: "8:00 PM", 21: "9:00 PM", 22: "10:00 PM", 23: "11:00 PM"
}

# Aggregate data: Count crashes per hour for each year
df_grouped = df_all.groupby(["Year", "CRASH_HOUR"]).size().reset_index(name="Count")

# Ensure all hours (0-23) are included for every year
df_complete = pd.DataFrame()
for year in years:
    df_year = df_grouped[df_grouped["Year"] == year] 
    df_year = all_hours.merge(df_year, on="CRASH_HOUR", how="left").fillna(0)
    df_year["Year"] = year
    df_complete = pd.concat([df_complete, df_year])

df_complete["Count"] = df_complete["Count"].astype(int)

df_complete["Time_Label"] = df_complete["CRASH_HOUR"].map(hour_to_time)

# Create an empty figure
fig = go.Figure()

# Store visibility settings
buttons = []

# Add a trace for each year
for year in years:
    df_year = df_complete[df_complete["Year"] == year] 

    fig.add_trace(go.Pie(
        labels=df_year["Time_Label"],  
        values=df_year["Count"],  
        name=str(year),
        visible=True
    ))

# Create dropdown buttons for year selection
buttons.append({
    "label": "All Years",
    "method": "update",
    "args": [{"visible": [True] * len(years)}, {"title": "Car Crashes by Time of Day (All Years)"}]
})

for i, year in enumerate(years):
    visibility = [False] * len(years)
    visibility[i] = True

    buttons.append({
        "label": str(year),
        "method": "update",
        "args": [{"visible": visibility}, {"title": f"Car Crashes by Time of Day in {year}"}]
    })
    
# Update layout with dropdown menu
fig.update_layout(
    title="Car Crashes by Time of Day",
    updatemenus=[{
        "buttons": buttons,
        "direction": "down",
        "showactive": True,
    }]
)
fig.show()
